# Chapter 2: The Product That Does Everything

## The geometric product $\mathbf{ab} = \mathbf{a} \cdot \mathbf{b} + \mathbf{a} \wedge \mathbf{b}$

Standard linear algebra gives you two ways to multiply vectors:

- The **dot product** $\mathbf{a} \cdot \mathbf{b}$ — a scalar that measures alignment. Works in any dimension.
- The **cross product** $\mathbf{a} \times \mathbf{b}$ — a vector perpendicular to both inputs. But this *only works in $\mathbb{R}^3$*.

If your vectors live in $\mathbb{R}^{256}$ (as transformer hidden states do), the cross product is undefined. You have the dot product and... nothing else. Half the geometric information — the *plane* the vectors span, the *oriented area* of their parallelogram — is simply inaccessible.

**Geometric Algebra** (GA) solves this by replacing both products with a single, universal operation: the **geometric product**. For two vectors $\mathbf{a}$ and $\mathbf{b}$:

$$\mathbf{a}\mathbf{b} = \underbrace{\mathbf{a} \cdot \mathbf{b}}_{\text{grade-0 (scalar)}} + \underbrace{\mathbf{a} \wedge \mathbf{b}}_{\text{grade-2 (bivector)}}$$

The scalar part is the familiar dot product. The new object $\mathbf{a} \wedge \mathbf{b}$ is called the **outer product** or **wedge product** — a **bivector** that encodes the oriented plane spanned by $\mathbf{a}$ and $\mathbf{b}$ and the area of the parallelogram they form. This works in *any* dimension.

In this chapter, we will compute geometric products, examine how they decompose into scalar and bivector parts, and apply them to transformer hidden states.

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from layer_time_ga.algebra import (
    geometric_product_vectors,
    grade_decomposition,
    bivector_from_skew,
)

import ltg_ga

print("Imports OK")

## The Geometric Product

For two vectors $\mathbf{a}, \mathbf{b} \in \mathbb{R}^k$, the geometric product is:

$$\mathbf{a}\mathbf{b} = \mathbf{a} \cdot \mathbf{b} + \mathbf{a} \wedge \mathbf{b}$$

This is a sum of objects of *different grades*:

- **Grade 0 (scalar):** $\mathbf{a} \cdot \mathbf{b} = \|\mathbf{a}\| \, \|\mathbf{b}\| \cos\theta$. This is the inner product from Chapter 1.
- **Grade 2 (bivector):** $\mathbf{a} \wedge \mathbf{b}$. Its magnitude is $\|\mathbf{a}\| \, \|\mathbf{b}\| \sin\theta$ — the area of the parallelogram spanned by the two vectors. It also encodes *which plane* they span.

The two parts are complementary:
- **Parallel vectors** ($\theta = 0$): The scalar part is maximal ($\cos 0 = 1$), and the bivector vanishes ($\sin 0 = 0$). Parallel vectors define no unique plane.
- **Perpendicular vectors** ($\theta = 90°$): The scalar part vanishes ($\cos 90° = 0$), and the bivector is maximal ($\sin 90° = 1$). Perpendicular vectors define a unique plane with maximum area.

Let us verify these properties computationally.

In [ ]:
# --- Perpendicular vectors: scalar = 0, bivector != 0 ---

a = np.array([1.0, 0.0, 0.0, 0.0])
b = np.array([0.0, 1.0, 0.0, 0.0])

gp = geometric_product_vectors(a, b)

print("=== Perpendicular vectors (theta = 90 degrees) ===")
print(f"a = {a}")
print(f"b = {b}")
print(f"Scalar part (a . b):     {gp['scalar']:.4f}   <-- zero, as expected")
print(f"Bivector norm ||a ^ b||: {gp['bivector'].norm:.4f}   <-- nonzero!")
print(f"Bivector matrix (skew-symmetric):")
print(gp['bivector'].matrix)
print()
print("The bivector is a 4x4 skew-symmetric matrix.")
print("The nonzero entries in the (0,1) and (1,0) positions tell us")
print("the plane of rotation is the e1-e2 plane.")

In [ ]:
# --- Parallel vectors: scalar != 0, bivector = 0 ---

a = np.array([1.0, 0.0, 0.0, 0.0])
b_parallel = np.array([3.0, 0.0, 0.0, 0.0])

gp_par = geometric_product_vectors(a, b_parallel)

print("=== Parallel vectors (theta = 0 degrees) ===")
print(f"a = {a}")
print(f"b = {b_parallel}")
print(f"Scalar part (a . b):     {gp_par['scalar']:.4f}   <-- nonzero!")
print(f"Bivector norm ||a ^ b||: {gp_par['bivector'].norm:.6f}   <-- zero, as expected")
print()
print("Parallel vectors have maximum scalar overlap and zero bivector.")
print("There is no unique plane defined by two parallel vectors.")

## The Outer Product: An Oriented Plane

The wedge (outer) product $\mathbf{a} \wedge \mathbf{b}$ produces a **bivector** — a fundamentally new kind of geometric object. It represents:

1. **A plane**: the 2D subspace spanned by $\mathbf{a}$ and $\mathbf{b}$.
2. **An area**: $\|\mathbf{a} \wedge \mathbf{b}\| = \|\mathbf{a}\| \, \|\mathbf{b}\| \sin\theta$, the area of the parallelogram formed by the two vectors.
3. **An orientation**: $\mathbf{a} \wedge \mathbf{b} = -\mathbf{b} \wedge \mathbf{a}$ — swapping the order flips the sign, like a "circulation direction" in the plane.

As the angle $\theta$ between two unit vectors sweeps from $0$ to $\pi/2$:
- The **scalar part** ($\cos\theta$) decreases from 1 to 0
- The **bivector norm** ($\sin\theta$) increases from 0 to 1

At $\theta = \pi/4$ (45 degrees), the two parts are equal: $\cos 45° = \sin 45° = 1/\sqrt{2}$. This is the "balanced" point where the geometric product carries equal weight in alignment and plane information.

Let us sweep through angles and visualize this tradeoff.

In [ ]:
# Sweep theta from 0 to pi/2 and track scalar vs bivector parts

thetas = np.linspace(0, np.pi / 2, 50)
scalar_parts = []
bivector_norms = []

e1 = np.array([1.0, 0.0, 0.0, 0.0])

for theta in thetas:
    # Construct b at angle theta from e1 in the e1-e2 plane
    b = np.cos(theta) * np.array([1.0, 0.0, 0.0, 0.0]) + \
        np.sin(theta) * np.array([0.0, 1.0, 0.0, 0.0])
    gp = geometric_product_vectors(e1, b)
    scalar_parts.append(gp['scalar'])
    bivector_norms.append(gp['bivector'].norm)

scalar_parts = np.array(scalar_parts)
bivector_norms = np.array(bivector_norms)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.degrees(thetas), scalar_parts, color='#4477AA', linewidth=2.5,
        label=r'Scalar part $\mathbf{a} \cdot \mathbf{b} = \cos\theta$')
ax.plot(np.degrees(thetas), bivector_norms, color='#EE6677', linewidth=2.5,
        label=r'Bivector norm $\|\mathbf{a} \wedge \mathbf{b}\| \propto \sin\theta$')
ax.axvline(45, color='grey', linestyle='--', alpha=0.5, label=r'$\theta = 45°$ (balanced)')
ax.set_xlabel(r'Angle $\theta$ (degrees)', fontsize=12)
ax.set_ylabel('Magnitude', fontsize=12)
ax.set_title('Scalar vs Bivector Parts of the Geometric Product', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 90)
ax.set_ylim(-0.05, 1.15)
fig.tight_layout()
plt.show()

print(f"At theta = 0:   scalar = {scalar_parts[0]:.3f}, bivector = {bivector_norms[0]:.3f}")
print(f"At theta = 45:  scalar = {scalar_parts[24]:.3f}, bivector = {bivector_norms[24]:.3f}")
print(f"At theta = 90:  scalar = {scalar_parts[-1]:.3f}, bivector = {bivector_norms[-1]:.3f}")

## In Matrix Form

How do we represent a bivector concretely? In $\mathbb{R}^k$, a bivector $\mathbf{a} \wedge \mathbf{b}$ can be stored as a $k \times k$ **skew-symmetric matrix**:

$$(\mathbf{a} \wedge \mathbf{b})_{ij} = a_i b_j - a_j b_i$$

A skew-symmetric matrix $A$ satisfies $A^T = -A$. Its diagonal is always zero, so the independent entries lie in the upper triangle. For a $k \times k$ matrix, this gives:

$$\text{number of independent components} = \frac{k(k-1)}{2}$$

These $k(k-1)/2$ numbers are the coordinates of the bivector in the basis of elementary bivectors $\{e_i \wedge e_j : i < j\}$. In $\mathbb{R}^4$, a bivector has $4 \cdot 3 / 2 = 6$ independent components. In $\mathbb{R}^{256}$, a bivector has $256 \cdot 255 / 2 = 32{,}640$ components — a rich geometric object indeed.

The `bivector_from_skew` function wraps any skew-symmetric matrix as a `Bivector` object with convenient properties.

In [ ]:
# Create a bivector from a skew-symmetric matrix

# Start with an arbitrary 5x5 matrix and extract its skew part
M = np.random.randn(5, 5)
A = 0.5 * (M - M.T)   # force skew-symmetry

biv = bivector_from_skew(A)

print("=== Bivector from a skew-symmetric matrix ===")
print(f"Dimension k = {biv.dim}")
print(f"Independent components: k(k-1)/2 = {biv.n_components}")
print(f"Frobenius norm: {biv.norm:.4f}")
print()

# Verify skew-symmetry: A^T should equal -A
print("Verification: is it skew-symmetric?")
print(f"  max |A + A^T| = {np.max(np.abs(biv.matrix + biv.matrix.T)):.2e}  (should be ~0)")
print()

print("Matrix representation:")
print(np.round(biv.matrix, 4))
print()

print(f"Upper-triangle components ({biv.n_components} values):")
print(np.round(biv.components(), 4))

## Grade Decomposition

Any square matrix $M$ can be split into a **symmetric** part and a **skew-symmetric** part:

$$M = \underbrace{\frac{M + M^T}{2}}_{\text{symmetric (grade-0)}} + \underbrace{\frac{M - M^T}{2}}_{\text{skew-symmetric (grade-2)}}$$

In geometric algebra language:
- The **symmetric part** corresponds to a **grade-0** (scalar/metric) quantity. It describes stretching and scaling — how much the transformation changes lengths.
- The **skew-symmetric part** corresponds to a **grade-2** (bivector) quantity. It describes rotation — how the transformation reorients directions in specific planes.

This decomposition is exact: the original matrix can be perfectly reconstructed from its two parts. The `grade_decomposition` function performs this split and returns both parts along with their norms.

This idea becomes powerful when applied to **layer transition matrices** in a transformer: the grade-0 part tells us about metric deformation (stretching), while the grade-2 part tells us about rotation (reorientation of the representation).

In [ ]:
# Grade decomposition of a random matrix

np.random.seed(42)
M = np.random.randn(6, 6)

decomp = grade_decomposition(M)

print("=== Grade Decomposition ===")
print(f"Original matrix M: shape {M.shape}")
print(f"  ||M||_F = {np.linalg.norm(M, 'fro'):.4f}")
print()
print(f"Grade-0 (symmetric part):")
print(f"  ||S||_F = {decomp['grade_0_norm']:.4f}")
print(f"  Is symmetric? max|S - S^T| = {np.max(np.abs(decomp['grade_0'] - decomp['grade_0'].T)):.2e}")
print()
print(f"Grade-2 (bivector / skew-symmetric part):")
print(f"  ||A||_F = {decomp['grade_2_norm']:.4f}")
print(f"  Bivector components: {decomp['grade_2'].n_components}")
print()

# Verify reconstruction: M = S + A
reconstructed = decomp['grade_0'] + decomp['grade_2'].matrix
reconstruction_error = np.max(np.abs(M - reconstructed))
print(f"Reconstruction check: max|M - (S + A)| = {reconstruction_error:.2e}")
print()

# The ratio of grade-0 to grade-2 tells us about the "character" of the matrix
ratio = decomp['grade_0_norm'] / (decomp['grade_2_norm'] + 1e-12)
print(f"Grade ratio ||S|| / ||A|| = {ratio:.3f}")
if ratio > 2:
    print("  --> Dominated by stretching (metric deformation)")
elif ratio < 0.5:
    print("  --> Dominated by rotation (bivector)")
else:
    print("  --> Mixed character (both stretching and rotation)")

## In the Transformer

Now let us apply the geometric product to actual transformer hidden states. We will take two token representations $H^{(l, t_1)}$ and $H^{(l, t_2)}$ at the same layer and compute their geometric product.

The result tells us:
- **Scalar part** ($\mathbf{a} \cdot \mathbf{b}$): how aligned these two token representations are — the same information cosine similarity gives us.
- **Bivector part** ($\mathbf{a} \wedge \mathbf{b}$): the oriented plane in $\mathbb{R}^k$ that the two representations span, and the area of the parallelogram they form. This is the geometric information that cosine similarity *discards*.

The bivector part is particularly interesting because it tells us about the **subspace of disagreement** between two tokens. If the model represents "France" and "Japan" slightly differently, the bivector tells us *in which directions* they differ — not just how much.

In [ ]:
# Load a model and analyse a prompt
model = ltg_ga.load_model("Qwen/Qwen2.5-7B")

prompt = "The capital of France is Paris and the capital of Japan is"
result = ltg_ga.analyse(prompt, model=model, compute_dependency=False)

H = result.H_whitened  # (L, T, k)
L, T, k = H.shape
tokens = result.tokens

print(f"Tokens: {tokens}")
print(f"Grid: {L} layers x {T} tokens x {k} whitened dims")

# Pick two token positions to compare
# Find tokens of interest
print("\nToken index mapping:")
for i, tok in enumerate(tokens):
    print(f"  [{i}] '{tok}'")

In [ ]:
# Compute the geometric product of two token hidden states at a middle layer

layer_idx = L // 2
t1, t2 = 0, min(T - 1, 5)  # first token vs a later token

h1 = H[layer_idx, t1]   # (k,) vector
h2 = H[layer_idx, t2]   # (k,) vector

gp = geometric_product_vectors(h1, h2)

# Also compute cosine similarity for comparison
cos_sim = np.dot(h1, h2) / (np.linalg.norm(h1) * np.linalg.norm(h2) + 1e-12)
theta = np.arccos(np.clip(cos_sim, -1, 1))

print(f"=== Geometric Product of Hidden States ===")
print(f"Token 1: '{tokens[t1]}' (position {t1})")
print(f"Token 2: '{tokens[t2]}' (position {t2})")
print(f"Layer: {layer_idx}")
print(f"Whitened dimension: k = {k}")
print()
print(f"--- Scalar Part (grade-0) ---")
print(f"  a . b = {gp['scalar']:.4f}")
print(f"  ||a|| = {np.linalg.norm(h1):.4f}")
print(f"  ||b|| = {np.linalg.norm(h2):.4f}")
print(f"  Cosine similarity = {cos_sim:.4f}")
print(f"  Angle theta = {np.degrees(theta):.1f} degrees")
print()
print(f"--- Bivector Part (grade-2) ---")
print(f"  ||a ^ b|| = {gp['bivector'].norm:.4f}")
print(f"  Independent components: {gp['bivector'].n_components}")
print(f"  Bivector matrix shape: {gp['bivector'].matrix.shape}")
print()

# The Pythagorean-like identity: |ab|^2 = (a.b)^2 + |a^b|^2/2
# For unit vectors: cos^2(theta) + sin^2(theta) = 1
area = np.linalg.norm(h1) * np.linalg.norm(h2) * np.sin(theta)
print(f"--- Geometric Interpretation ---")
print(f"  Parallelogram area (||a|| ||b|| sin theta): {area:.4f}")
print(f"  The scalar part tells us: the vectors are {cos_sim:.1%} aligned")
print(f"  The bivector part tells us: they span a plane with area {area:.1f}"
      f" in {k}-dimensional space")

## Exercises

**Exercise 2.1 — Anticommutativity of the wedge product.**
Compute `geometric_product_vectors(h1, h2)` and `geometric_product_vectors(h2, h1)` for the same pair of hidden states. Verify that the scalar parts are identical ($\mathbf{a} \cdot \mathbf{b} = \mathbf{b} \cdot \mathbf{a}$) but the bivector matrices are negatives of each other ($\mathbf{a} \wedge \mathbf{b} = -\mathbf{b} \wedge \mathbf{a}$). Why does this anticommutativity make geometric sense? (Hint: think about orientation.)

**Exercise 2.2 — Grade decomposition across layers.**
For a random pair of tokens, compute `grade_decomposition` of the outer product matrix $\mathbf{a} \mathbf{b}^T$ at layers 2, 14, and 26. Track the ratio $\|S\|_F / \|A\|_F$ (symmetric norm / skew norm). Does the balance between stretching and rotation change across layers? What might this tell you about how the transformer processes information?

**Exercise 2.3 — Bivector principal planes.**
Take the bivector from the geometric product of two hidden states and call `gp['bivector'].principal_planes(n_planes=3)`. This decomposes the bivector into its dominant rotation planes. How much of the total bivector norm is concentrated in the first principal plane? In $\mathbb{R}^{256}$ there are over 32,000 possible plane directions — does the bivector use all of them equally, or is it concentrated in a low-dimensional subspace?